# Climate Change Trend Analysis and Forecasting

**Author:** Sauparna Sarkar  
**Team:** *(fill in your names)*

---

## Table of Contents

- [Week 1 — Data Acquisition, Exploration and Understanding](#week1)
  - [1.0 Imports & Constants](#10-imports)
  - [1.1 Data Loading](#11-data-loading)
  - [1.2 Data Profiling](#12-data-profiling)
  - [1.3 Exploratory Data Analysis](#13-eda)
- [Week 2 — Feature Engineering](#week2)
  - [2.1 Time-Based Features](#21-time-based)
  - [2.2 Lag Features](#22-lag)
  - [2.3 Per-Capita and Intensity Features](#23-intensity)
  - [2.4 Growth Rate Features](#24-growth)
  - [2.5 Final Feature Dataset](#25-final)
- [Week 3 — Baseline ML Models: Regression](#week3)
  - [3.1 Problem Framing](#31-framing)
  - [3.2 Train-Test Split](#32-split)
  - [3.3 Naive Baseline](#33-naive)
  - [3.4 Linear Regression](#34-lr)
  - [3.5 Random Forest Regressor — Per Country](#35-rf-pc)
  - [3.6 Random Forest Regressor — Pooled](#36-rf-pooled)
  - [3.7 Model Comparison Table](#37-comparison)
- [Week 4 — Time-Series Forecasting with ETS(A,Ad,N)](#week4)
  - [4.1 Concept Introduction](#41-concept)
  - [4.2 Model Fitting](#42-model-fitting)
  - [4.3 Forecasting to 2043](#43-forecasting)
  - [4.4 Trend Interpretation](#44-trend)
  - [4.5 Forecast Summary Table](#45-forecast-table)
  - [4.6 Model Validation](#46-validation)
- [Week 5 — Scenario Analysis *(Optional)*](#week5)
- [Week 6 — Notebook Finalisation](#week6)
- [Week 7 — Final Presentation](#week7)

---
<a id='week1'></a>
# Week 1 — Data Acquisition, Exploration and Understanding

**Learning Objective:** Understand the structure and content of GHG datasets and produce a clean, profiled dataset ready for analysis.

> **Notebook standard (apply throughout):** Every code cell must be preceded by a markdown cell explaining *what the code does and why*. All charts must have titles, axis labels, and legends. All variable names must be descriptive — no single-letter names except loop counters.


<a id='10-imports'></a>
### 1.0 Imports

Import all libraries needed for the entire project here. Add to this cell as you progress through the weeks.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

### Project Constants

The cell below is **pre-filled** with shared constants used throughout the notebook. Do not modify these values without discussing the change first.

> **Important OWID naming note:** The dataset uses `"United States"` (not `"USA"`) and `"United Kingdom"` (not `"UK"`). Use these exact strings when filtering.


In [ ]:
# ── Project constants ─────────────────────────────────────────────────────
OWID_URL = "https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv"

COUNTRIES = [
    "China", "United States", "India", "Russia", "Japan",
    "Germany", "Brazil", "United Kingdom", "South Africa", "Australia",
]
# Note: OWID uses "United States" (not "USA") and "United Kingdom" (not "UK")

TRAIN_CUTOFF = 2018   # train: 1990–2018  |  test: 2019–2023
FORECAST_END = 2043   # 20 years beyond 2023

NON_SOVEREIGN = [
    # Continental / regional aggregates (OWID)
    "World", "Asia", "Europe", "Africa", "North America", "South America",
    "Oceania",
    # Continental / regional aggregates (GCP variants)
    "Africa (GCP)", "Asia (GCP)", "Europe (GCP)",
    "North America (GCP)", "South America (GCP)", "Oceania (GCP)",
    "Central America (GCP)", "Middle East (GCP)",
    # Sub-regional exclusion variants
    "Asia (excl. China and India)",
    "Europe (excl. EU-27)", "Europe (excl. EU-28)",
    "North America (excl. USA)",
    # European Union aggregates
    "European Union (27)", "European Union (28)",
    # Income / development groupings
    "High-income countries", "Low-income countries",
    "Upper-middle-income countries", "Lower-middle-income countries",
    "Least developed countries (Jones et al.)",
    # OECD / Non-OECD groupings
    "OECD (GCP)", "OECD (Jones et al.)", "Non-OECD (GCP)",
    # International transport (components — "International transport" does not exist in OWID)
    "International aviation", "International shipping",
    # Special / historical entries
    "Kuwaiti Oil Fires", "Kuwaiti Oil Fires (GCP)",
    "Ryukyu Islands (GCP)",
]

GAS_COLUMNS = {
    "CO\u2082":                "co2",
    "Methane (CH\u2084)":      "methane",
    "Nitrous Oxide (N\u2082O)": "nitrous_oxide",
}

<a id='11-data-loading'></a>
### 1.1 Data Loading

**What to do:** Load the OWID CO₂ dataset from `OWID_URL` (or `data/owid-co2-data.csv` if downloaded locally). Display the first 10 rows, all column names, data types, and the shape of the DataFrame.

**Key columns:**

| Column | Description |
|--------|-------------|
| `country` | Country or region name |
| `year` | Year of observation |
| `co2` | Annual CO₂ emissions (million tonnes) |
| `co2_per_capita` | CO₂ per person (tonnes/person) |
| `methane` | Methane emissions, CO₂-equivalent (MtCO₂e) |
| `nitrous_oxide` | N₂O emissions, CO₂-equivalent (MtCO₂e) |
| `total_ghg` | Total GHG emissions (MtCO₂e) |
| `gdp` | GDP in international dollars |
| `population` | Population |


In [ ]:
import urllib.request

LOCAL_PATH = "../data/owid-co2-data.csv"

if os.path.exists(LOCAL_PATH):
    df_raw = pd.read_csv(LOCAL_PATH)
    print(f"Loaded from local file: {LOCAL_PATH}")
else:
    print("Local file not found — fetching from OWID URL (~50 MB, may take a moment)...")
    print("Tip: if this fails, download manually — see README §4 or run:")
    print(f"  curl -o data/owid-co2-data.csv {OWID_URL}")
    urllib.request.urlretrieve(OWID_URL, LOCAL_PATH)   # saves raw bytes; avoids corrupt-on-interrupt risk of parse→to_csv
    print(f"Saved to {LOCAL_PATH}")
    df_raw = pd.read_csv(LOCAL_PATH)

print(f"\nShape: {df_raw.shape}")
print(f"\nColumn names:\n{df_raw.columns.tolist()}")
print(f"\nData types:\n{df_raw.dtypes.to_string()}")
display(df_raw.head(10))

**Observation:** The OWID dataset contains over 50,000 rows spanning more than 200 countries and regions, with annual records from as early as the 18th century through to 2023. It includes a rich mix of emissions columns (`co2`, `methane`, `nitrous_oxide`, `total_ghg`), economic indicators (`gdp`, `gdp_per_capita`), and population data — many of which have significant missing values for earlier years and smaller nations. Notably, the dataset mixes sovereign countries with regional aggregates (e.g. "World", "Asia", "European Union") that will need to be excluded before modelling.

<a id='12-data-profiling'></a>
### 1.2 Data Profiling

**Tasks:**
1. Report null values per column **as a percentage of total rows** (sorted descending).
2. Identify which countries and years have the most complete data coverage.
3. Filter to keep only rows where `year >= 1990` and `country` is not in `NON_SOVEREIGN`.


In [ ]:
null_pct = (df_raw.isnull().sum() / len(df_raw) * 100).sort_values(ascending=False)

print("Null values as % of total rows (columns with any nulls):")
print(null_pct[null_pct > 0].round(1).to_string())

**Filtering decisions:** We retain only rows where `year >= 1990` for two reasons: (1) emissions reporting becomes substantially more complete and standardised from 1990 onward, and (2) 1990 is the conventional UNFCCC baseline year, making it the natural starting point for policy-relevant analysis. We exclude entries in `NON_SOVEREIGN` (regional aggregates like "World", "Asia", "European Union") because they are arithmetic sums of their constituent countries — including them would double-count emissions when computing global or multi-country totals. After filtering we retain approximately 15% of the original rows (the OWID dataset extends back to ~1750, so pre-1990 data accounts for the majority of rows excluded), covering 220 sovereign nations from 1990 to 2024.

In [ ]:
df_filtered = df_raw[
    (df_raw['year'] >= 1990) & (~df_raw['country'].isin(NON_SOVEREIGN))
].copy()

print(f"Rows before filtering: {len(df_raw):,}")
print(f"Rows after filtering:  {len(df_filtered):,} ({len(df_filtered)/len(df_raw)*100:.1f}% retained)")
print(f"Year range: {df_filtered['year'].min()} to {df_filtered['year'].max()}")
print(f"Unique countries: {df_filtered['country'].nunique()}")

key_cols = ['co2', 'co2_per_capita', 'total_ghg', 'methane', 'nitrous_oxide', 'gdp', 'population']

coverage = (
    df_filtered[key_cols + ['country']]
    .groupby('country')[key_cols]
    .apply(lambda x: x.notna().mean() * 100, include_groups=False)
    .mean(axis=1)
    .sort_values(ascending=False)
)

print("\nTop 10 sovereign countries by average data coverage across key columns (%):")
print(coverage.head(10).round(1).to_string())

<a id='13-eda'></a>
### 1.3 Exploratory Data Analysis (EDA)

Produce the three charts below. Every chart must have a **title, labelled axes, and a legend**. Write a 3–5 sentence observation after each one.


#### Chart 1 — Global CO₂ Emissions Trend (1990 to latest year)

**What to do:** Sum `co2` across all sovereign nations per year, then plot as a line chart.


In [ ]:
global_co2 = df_filtered.groupby('year')['co2'].sum().reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(global_co2['year'], global_co2['co2'], linewidth=2, color='steelblue')
ax.set_title('Global CO₂ Emissions (Sovereign Nations), 1990–Present')
ax.set_xlabel('Year')
ax.set_ylabel('CO₂ Emissions (MtCO₂)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

**Observation:** Global CO₂ emissions rose steadily from around 23,000 MtCO₂ in 1990 to around 36,000 MtCO₂ in 2019, with emissions continuing to rise to a peak of roughly 37,400 MtCO₂ in 2024, driven primarily by rapid industrialisation in Asia. Two clear inflection points are visible: a brief dip around 2008–2009 coinciding with the global financial crisis, and a sharper drop in 2020 caused by COVID-19 pandemic-related lockdowns and reduced economic activity. Emissions rebounded quickly in 2021 as economies reopened, suggesting the pandemic effect was transient rather than structural. The overall trend remains upward, though the rate of growth has moderated in the 2010s compared to the 2000s, likely reflecting improving energy efficiency and growing renewable energy deployment in major economies.

#### Chart 2 — CO₂ Trends for Top 5 Emitting Countries

**What to do:** Filter `df_filtered` to China, USA ("United States"), India, Russia, and Japan. Plot a multi-line chart with one line per country.


In [ ]:
TOP5 = ['China', 'United States', 'India', 'Russia', 'Japan']
df_top5 = df_filtered[df_filtered['country'].isin(TOP5)].copy().sort_values(['country', 'year'])

fig, ax = plt.subplots(figsize=(10, 5))
for country in TOP5:
    data = df_top5[df_top5['country'] == country]
    ax.plot(data['year'], data['co2'], label=country, linewidth=2)

ax.set_title('CO₂ Emissions — Top 5 Emitting Countries, 1990–Present')
ax.set_xlabel('Year')
ax.set_ylabel('CO₂ Emissions (MtCO₂)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

**Observation:** China's trajectory stands out dramatically — it more than tripled its CO₂ output between 1990 and 2014, overtaking the United States around 2006 to become the world's largest emitter by a wide margin. The United States shows a broadly flat-to-declining trend since the mid-2000s, reflecting a shift from coal to natural gas and growing renewable energy adoption. India shows a steady, accelerating upward trajectory throughout the period, reflecting its rapid economic growth and expanding electricity demand. Russia and Japan both remain relatively flat over the full period, with Russia's emissions dipping sharply in the early 1990s following the post-Soviet economic contraction. These divergent patterns illustrate how emissions trajectories are shaped by the intersection of economic growth stage, energy mix, and policy environment.

#### Chart 3 — Share of Global GHG by Gas Type per Decade

**What to do:** Add a `decade` column (`(df['year'] // 10) * 10`), group by decade, sum CO₂/CH₄/N₂O columns across all countries, normalise to percentages, and plot as a stacked bar chart.


In [ ]:
gas_cols = ['co2', 'methane', 'nitrous_oxide']

# Note: groupby().sum() treats NaN as 0 by default. Countries that do not report
# methane or N₂O contribute 0 to those columns but their full CO₂ to the total,
# which slightly overstates CO₂'s share. The 2020s bar covers only 2020–2023
# (4 years) — proportions are still valid, but absolute totals would be lower.
decade_key = (df_filtered['year'] // 10) * 10   # Series used as groupby key; avoids mutating df_filtered
decade_sums = df_filtered.groupby(decade_key)[gas_cols].sum()
decade_pct = decade_sums.div(decade_sums.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(9, 5))
decade_pct.plot(
    kind='bar', stacked=True, ax=ax,
    color=['steelblue', 'darkorange', 'seagreen'],
    edgecolor='white', width=0.6
)
ax.set_title('Share of Global GHG Emissions by Gas Type per Decade')
ax.set_xlabel('Decade')
ax.set_ylabel('Share (%)')
ax.set_xticklabels(
    [f"{int(d)}s" + (" *" if d == 2020 else "") for d in decade_pct.index],
    rotation=0
)
ax.legend(['CO₂', 'Methane (CH₄)', 'Nitrous Oxide (N₂O)'], loc='lower right')
ax.grid(axis='y', alpha=0.3)
ax.text(0.99, 0.01, '* 2020s = 2020–2023 only', transform=ax.transAxes,
        ha='right', va='bottom', fontsize=8, color='grey')
plt.tight_layout()
plt.show()

**Observation:** CO₂ consistently dominates the GHG mix, accounting for approximately 70–75% of total emissions across all decades, with its share showing a modest upward trend over time as fossil fuel combustion has grown faster than agricultural sources. Methane (CH₄) is the second-largest contributor at around 20–25%, arising primarily from livestock, rice cultivation, and natural gas leakage; its share has remained relatively stable. Nitrous oxide (N₂O), largely driven by synthetic fertiliser use and livestock waste, accounts for the remaining 5–8% and has also held roughly constant. The stability of the gas-type composition across decades suggests that while total emissions have grown substantially, the underlying structure of global GHG production has not shifted dramatically — decarbonisation of the electricity sector (which primarily reduces CO₂) would need to be accompanied by agricultural methane reductions to substantially change this mix.

### Week 1 Summary

The OWID CO₂ dataset is a rich, wide-format panel covering ~200 sovereign nations from 1990 to 2023, with 70+ columns spanning emissions by gas type, per-capita and intensity measures, energy mix, GDP, and population. Many columns — particularly economic and non-CO₂ gas columns — have significant missingness for earlier years and smaller nations, so filtering to post-1990 data and excluding regional aggregates was essential before any analysis. The global CO₂ trend reveals steady growth punctuated by two clear demand shocks (2008–09 and 2020), with the 2020 pandemic dip quickly reversed — suggesting that structural change rather than crisis-driven disruption is needed to bend the curve. Disaggregating by country exposes the critical role of China, whose tripling of output since 1990 accounts for most of global emissions growth, while the USA and developed economies show plateaus or modest declines. Finally, the gas-type breakdown shows that CO₂ dominates and its share is growing, while methane and N₂O shares are stable — a finding that will inform our modelling focus on CO₂ as the primary target variable in the weeks ahead.

---
> **Commit before your Week 1 review session:**
> ```
> git add notebook/ghg_analysis.ipynb
> git commit -m "Week 1: data loading, profiling, and EDA complete"
> git push
> ```

---
<a id='week2'></a>
# Week 2 — Feature Engineering

**Learning Objective:** Transform raw emissions data into a structured, model-ready feature set.

Work from `df_filtered`. Apply all `groupby().transform()` operations to the full filtered dataset, then subset to `COUNTRIES` at the end (§2.5).


<a id='21-time-based'></a>
### 2.1 Time-Based Features

Add three columns:

| New Column | How to compute |
|------------|----------------|
| `decade` | `(df['year'] // 10) * 10` |
| `years_since_1990` | `df['year'] - 1990` — numeric time index for regression |
| `co2_5yr_rolling_mean` | `groupby('country')['co2'].transform(lambda x: x.rolling(5).mean())` |

> **Note:** The rolling mean will be `NaN` for the first 4 rows of each country. This is correct — do not fill.


In [ ]:
df = df_filtered.copy()   # working copy for all Week 2 mutations; keeps df_filtered clean

df.loc[:, 'decade']               = (df['year'] // 10) * 10
df.loc[:, 'years_since_1990']     = df['year'] - 1990
df.loc[:, 'co2_5yr_rolling_mean'] = (
    df.groupby('country')['co2']
    .transform(lambda x: x.rolling(5).mean())   # default min_periods=5: NaN until 5 consecutive non-null values
)

print(f"Shape after adding time features: {df.shape}")
n_nan = df['co2_5yr_rolling_mean'].isna().sum()
n_co2_nan = df['co2'].isna().sum()
print(f"\nNaN in co2_5yr_rolling_mean: {n_nan:,}")
print(f"  (at minimum 4 per country from window build-up; extra from {n_co2_nan:,} NaN in co2 itself)")

print("\nSample — China 1990–1997 (rolling mean builds up from 1994 onward):")
display(
    df[df['country'] == 'China'][['year', 'decade', 'years_since_1990', 'co2', 'co2_5yr_rolling_mean']]
    .head(8)
    .reset_index(drop=True)
)

<a id='22-lag'></a>
### 2.2 Lag Features

Create three lag columns per country using `groupby('country')['co2'].shift(n)`:

| Column | Shift |
|--------|-------|
| `co2_lag1` | `shift(1)` — previous year CO₂ |
| `co2_lag2` | `shift(2)` |
| `co2_lag3` | `shift(3)` |

**What is a lag feature?** A lag feature is the value of a variable from N prior time steps. `co2_lag1` for country C in year Y contains C's CO₂ emissions from year Y−1. Lag features capture **temporal autocorrelation** — emissions change slowly due to the inertia of energy infrastructure, industrial capacity, and economic growth, so last year's emissions are highly predictive of this year's. Providing the model with lag values avoids the need to explicitly model the time series dynamics in the regression framework.

**Key assumption:** Using lag features assumes that the relationship between past and future values is **stationary** over the training window — i.e., the same lagged correlation structure that held from 1990–2018 continues to hold during the 2019–2023 test period. This assumption is reasonable for slowly-evolving national emissions series but can be violated by structural shocks (e.g. the 2020 COVID-19 lockdowns).


In [ ]:
for lag in [1, 2, 3]:
    df.loc[:, f'co2_lag{lag}'] = df.groupby('country')['co2'].shift(lag)

print("Lag columns added: co2_lag1, co2_lag2, co2_lag3")
print("\nNaN counts (expected: lag × number of countries, plus propagation from NaN co2):")
for lag in [1, 2, 3]:
    print(f"  co2_lag{lag}: {df[f'co2_lag{lag}'].isna().sum():,}")

print("\nSample — United States 1990–1995 (lag columns shift emissions back N years):")
display(
    df[df['country'] == 'United States'][
        ['year', 'co2', 'co2_lag1', 'co2_lag2', 'co2_lag3']
    ].head(6).reset_index(drop=True)
)

<a id='23-intensity'></a>
### 2.3 Per-Capita and Intensity Features

**Tasks:**
1. **Verify `co2_per_capita`:** Cross-check against `co2 / population * 1e6` for at least 3 countries × 3 years.
2. **Create `ghg_intensity`:** `total_ghg / gdp` where both are non-null.
3. **Report missingness:** For how many country-year rows is `ghg_intensity` not computable?


In [ ]:
# ── Verify co2_per_capita ─────────────────────────────────────────────────────
print("Verification: co2_per_capita vs co2 / population * 1e6")
print("(population in OWID is persons; co2 is MtCO₂; dividing by 1e6 converts to tonnes/person)\n")

for country in ['China', 'United States', 'India']:
    sub = df[
        (df['country'] == country) & (df['year'].isin([2000, 2010, 2020]))
    ][['year', 'co2', 'population', 'co2_per_capita']].copy()
    sub.loc[:, 'computed'] = sub['co2'] / sub['population'] * 1e6
    sub.loc[:, 'abs_diff'] = (sub['co2_per_capita'] - sub['computed']).abs()
    print(f"  {country}:")
    display(sub.reset_index(drop=True))

# ── Create ghg_intensity ───────────────────────────────────────────────────────
df.loc[:, 'ghg_intensity'] = df['total_ghg'] / df['gdp']

n_miss  = df['ghg_intensity'].isna().sum()
n_total = len(df)
print(f"\nghg_intensity NaN: {n_miss:,} / {n_total:,} rows ({n_miss/n_total*100:.1f}%)")
print("Main causes: GDP missing for smaller nations and for most countries before ~1995")

**Report — `ghg_intensity` availability:** `ghg_intensity` (= `total_ghg / gdp`) is unavailable for roughly 70% of rows in `df`. The primary driver is that `gdp` is missing for most smaller and lower-income nations across the full period, and for nearly all countries before 1990 (though we've already filtered that). Among the 10 focus countries, GDP coverage is generally good from the mid-1990s onward, but `total_ghg` itself has ~24% NaN globally, adding a second source of missingness.

For use as a model feature in Week 3: any regression that includes `ghg_intensity` will drop all rows where it is NaN, significantly shrinking the training set for countries with data gaps. The safest approach is to treat it as an **optional supplementary feature** and report separately on models with and without it, or to exclude it from the primary feature set used in §3.1.

<a id='24-growth'></a>
### 2.4 Growth Rate Features

Compute two year-on-year columns per country:

| Column | Formula |
|--------|--------|
| `co2_yoy_change` | `groupby('country')['co2'].diff()` |
| `co2_yoy_pct_change` | `groupby('country')['co2'].pct_change() * 100` |

Then identify:
- Top 5 countries by **highest average annual CO₂ growth rate** since 1990
- Top 5 countries by **largest total CO₂ reduction** since 1990


In [ ]:
df.loc[:, 'co2_yoy_change']     = df.groupby('country')['co2'].diff()
df.loc[:, 'co2_yoy_pct_change'] = df.groupby('country')['co2'].pct_change(fill_method=None) * 100

print("YoY columns added: co2_yoy_change, co2_yoy_pct_change")

# Top 5 by highest average annual growth rate (all sovereign nations)
avg_growth = (
    df.groupby('country')['co2_yoy_pct_change']
    .mean()
    .sort_values(ascending=False)
)
print("\nTop 5 countries (all sovereign nations) by average annual CO₂ growth rate (%) since 1990:")
display(avg_growth.head(5).round(2).to_frame('avg_annual_pct_growth'))

# Top 5 by largest total CO₂ reduction (1990 → latest year)
latest_year = df['year'].max()
co2_1990    = df[df['year'] == 1990].set_index('country')['co2']
co2_latest  = df[df['year'] == latest_year].set_index('country')['co2']
reduction   = (co2_1990 - co2_latest).dropna().sort_values(ascending=False)

print(f"\nTop 5 countries by total CO₂ reduction 1990 → {latest_year} (MtCO₂):")
display(reduction.head(5).round(1).to_frame('co2_reduction_MtCO2'))

**Findings:** The top growth-rate countries are dominated by small or rapidly industrialising nations — often island states or Gulf oil producers that expanded from a near-zero base, producing very high percentage growth rates despite modest absolute volumes. Among the 10 focus countries, **India** shows the highest average annual growth rate, consistent with its steady upward trajectory in the Week 1 Chart 2. **China** would rank near the top in absolute volume growth but its percentage growth rate has moderated as its base grew large.

For total CO₂ reductions, the UK and Germany consistently lead among the 10 focus countries, reflecting decades of coal phase-out, energy efficiency improvements, and renewable deployment. **Russia** also shows a large apparent reduction, but this largely reflects the economic collapse following the Soviet dissolution in the early 1990s rather than deliberate climate policy. These findings are consistent with the Week 1 multi-line chart, where Russia's sharp early-1990s dip is clearly visible.

<a id='25-final'></a>
### 2.5 Final Feature Dataset

Filter to `country` in `COUNTRIES` and keep these columns:

```
country, year, co2, co2_per_capita, years_since_1990, co2_5yr_rolling_mean,
co2_lag1, co2_lag2, co2_lag3, co2_yoy_pct_change, ghg_intensity
```

Expected shape: ~10 countries × 35 years ≈ 350 rows (varies with data availability).

Save as `data/ghg_features.csv`.


In [ ]:
feature_cols = [
    'country', 'year', 'co2', 'co2_per_capita',
    'years_since_1990', 'co2_5yr_rolling_mean',
    'co2_lag1', 'co2_lag2', 'co2_lag3',
    'co2_yoy_pct_change', 'ghg_intensity',
]

df_features = (
    df[df['country'].isin(COUNTRIES)][feature_cols]
    .sort_values(['country', 'year'])
    .reset_index(drop=True)
)

print(f"Shape: {df_features.shape}  "
      f"({df_features['country'].nunique()} countries × {df_features['year'].nunique()} years)")
print("\nNull counts per column:")
display(df_features.isnull().sum().to_frame('null_count'))

df_features.to_csv('../data/ghg_features.csv', index=False)
print("\nSaved to data/ghg_features.csv")
display(df_features.head(10))

### Week 2 Summary

Feature engineering transformed the raw OWID dataset into a model-ready 10-column feature set. The lag features (`co2_lag1/2/3`) and rolling mean (`co2_5yr_rolling_mean`) are the most complete — NaN appears only at the start of each country's series and wherever `co2` itself is missing. `ghg_intensity` has the highest missingness (~70%) due to sparse GDP data, and should be treated as optional in Week 3 models. Among the 10 focus countries, **India** shows the highest average CO₂ growth rate since 1990, while **Germany** and the **UK** lead in total absolute reductions — findings consistent with the Week 1 EDA charts. The final `ghg_features.csv` (~350 rows, 10 columns) is saved to `data/` and will serve as the single input for all Week 3 regression models.

---
> ```
> git add notebook/ghg_analysis.ipynb data/ghg_features.csv
> git commit -m "Week 2: feature engineering complete, ghg_features.csv added"
> git push
> ```

---
<a id='week3'></a>
# Week 3 — Baseline ML Models: Regression

**Learning Objective:** Train, evaluate, and compare supervised regression models to predict CO₂ emissions.

Load `data/ghg_features.csv` here so Week 3 can run independently.


In [ ]:
df_features = pd.read_csv('../data/ghg_features.csv')

<a id='31-framing'></a>
### 3.1 Problem Framing

**Prediction task:** Given features X for country C in year Y — lag emissions (`co2_lag1/2/3`), a 5-year rolling mean, a numeric time index (`years_since_1990`), and year-on-year % change — predict CO₂ emissions for year Y. The model is trained on 1990–2018 and evaluated on the 2019–2023 holdout. This is **regression** (not classification) because CO₂ is a continuous quantity, not a discrete category. `ghg_intensity` is excluded from the primary feature set due to ~70% missingness — including it would shrink the training set after `dropna` and introduce inconsistency across countries.

**Training strategy:** Three models are trained in Week 3 using two approaches. Linear Regression is trained per country (~25 rows each), adequate for a 6-feature linear model. Two Random Forest variants are trained: one per country (~25 rows each, §3.5) to expose the overfitting problem on small data, and one pooled across all 10 countries (~250 rows, §3.6) as the production approach. All models are evaluated per country on the 2019–2023 test set for direct comparison.


**Feature and target columns:**

```python
FEATURES = [
    'years_since_1990', 'co2_5yr_rolling_mean',
    'co2_lag1', 'co2_lag2', 'co2_lag3', 'co2_yoy_pct_change'
]
TARGET = 'co2'
```

> Drop rows with `NaN` in any of these columns before fitting — lag features produce NaN for earliest years.

<a id='32-split'></a>
### 3.2 Train-Test Split

- **Training:** `year <= TRAIN_CUTOFF` (1990–2018, ~28 samples per country after dropping NaN rows)
- **Test:** `year > TRAIN_CUTOFF` (2019–2023, 5 samples)

The 2019–2023 test window includes the COVID-19 emissions dip (2020) and recovery — a useful real-world stress test.

> **Do NOT use `train_test_split(shuffle=True)`** — random mixing leaks future values into training.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

FEATURES = [
    'years_since_1990', 'co2_5yr_rolling_mean',
    'co2_lag1', 'co2_lag2', 'co2_lag3', 'co2_yoy_pct_change',
]
TARGET = 'co2'

# Drop rows where any FEATURE or TARGET is NaN
df_model = (
    df_features[['country', 'year'] + FEATURES + [TARGET]]
    .dropna()
    .reset_index(drop=True)
)

train = df_model[df_model['year'] <= TRAIN_CUTOFF]
test  = df_model[df_model['year'] >  TRAIN_CUTOFF]

print(f"After dropna: {len(df_model)} rows  ({len(train)} train / {len(test)} test)")
print(f"Train years: {train['year'].min()}–{train['year'].max()}   "
      f"Test years: {test['year'].min()}–{test['year'].max()}")

counts = (
    df_model.groupby('country')['year']
    .agg(total='count', first_year='min', last_year='max')
    .join(train.groupby('country').size().rename('train_rows'))
    .join(test.groupby('country').size().rename('test_rows'))
)
display(counts)

**Why temporal splitting is essential:** Random splitting allows the model to train on rows from 2020 and then "predict" rows from 2010 — it sees the future during training. This leaks information and produces optimistically biased metrics. In a real forecasting scenario the model can only use past data, so the test set must strictly follow the training period in time.

If random splitting were used, MAE/RMSE would appear artificially low: lag features from later years would appear in the training set, effectively giving the model direct access to near-future values. The resulting error metrics would not reflect how well the model generalises to genuinely unseen future data.

<a id='33-naive'></a>
### 3.3 Naive Baseline

**Prediction:** next year CO₂ = this year CO₂ (i.e. `co2_lag1` from the test set).

Compute MAE and RMSE per country. Plot actual vs predicted for 3 countries.

```python
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
```


In [ ]:
PLOT_COUNTRIES = ['China', 'United States', 'Germany']

baseline_results = {}
for country in COUNTRIES:
    test_c = test[test['country'] == country]
    if test_c.empty:
        continue
    y_true = test_c[TARGET].values
    y_pred = test_c['co2_lag1'].values   # naive: last year's CO₂ = this year's prediction
    baseline_results[country] = {
        'Baseline MAE':  round(mean_absolute_error(y_true, y_pred), 1),
        'Baseline RMSE': round(np.sqrt(mean_squared_error(y_true, y_pred)), 1),
    }

print("Naive Baseline — test set 2019–present:")
display(pd.DataFrame(baseline_results).T)

# Actual vs predicted for 3 countries
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, country in zip(axes, PLOT_COUNTRIES):
    train_c = train[train['country'] == country]
    test_c  = test[test['country'] == country]
    ax.plot(train_c['year'], train_c[TARGET],    color='steelblue',  label='Train actuals', linewidth=1.5)
    ax.plot(test_c['year'],  test_c[TARGET],     color='darkorange', label='Test actuals',  linewidth=1.5)
    ax.plot(test_c['year'],  test_c['co2_lag1'], color='grey', linestyle='--', label='Baseline pred', linewidth=1.5)
    ax.set_title(country); ax.set_xlabel('Year'); ax.set_ylabel('CO₂ (MtCO₂)')
    ax.legend(fontsize=7); ax.grid(axis='y', alpha=0.3)

plt.suptitle('Naive Baseline: Actual vs Predicted CO₂ (test set)', fontsize=12)
plt.tight_layout()
plt.show()

<a id='34-lr'></a>
### 3.4 Linear Regression

Train `sklearn.linear_model.LinearRegression` per country. Compute MAE and RMSE. Plot predictions vs actuals for 3 countries. Print coefficients and identify the most influential feature.


In [ ]:
from sklearn.linear_model import LinearRegression

lr_results = {}
lr_models  = {}
lr_preds   = {}

for country in COUNTRIES:
    train_c = train[train["country"] == country]
    test_c  = test[test["country"] == country]
    if train_c.empty or test_c.empty:
        continue
    model  = LinearRegression()
    model.fit(train_c[FEATURES], train_c[TARGET])
    y_pred = model.predict(test_c[FEATURES])
    lr_results[country] = {
        "LR MAE":  round(mean_absolute_error(test_c[TARGET], y_pred), 1),
        "LR RMSE": round(np.sqrt(mean_squared_error(test_c[TARGET], y_pred)), 1),
    }
    lr_models[country] = model
    lr_preds[country]  = y_pred

print("Linear Regression — test set 2019–present:")
display(pd.DataFrame(lr_results).T)

# Actual vs predicted for 3 countries
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, country in zip(axes, PLOT_COUNTRIES):
    train_c = train[train["country"] == country]
    test_c  = test[test["country"] == country]
    ax.plot(train_c["year"], train_c[TARGET],   color="steelblue",  label="Train actuals", linewidth=1.5)
    ax.plot(test_c["year"],  test_c[TARGET],    color="darkorange", label="Test actuals",  linewidth=1.5)
    ax.plot(test_c["year"],  lr_preds[country], color="red", linestyle="--", label="LR pred", linewidth=1.5)
    ax.set_title(country); ax.set_xlabel("Year"); ax.set_ylabel("CO₂ (MtCO₂)")
    ax.legend(fontsize=7); ax.grid(axis="y", alpha=0.3)

plt.suptitle("Linear Regression: Actual vs Predicted CO₂ (test set)", fontsize=12)
plt.tight_layout()
plt.show()

# Coefficient plot — China
coef_s = pd.Series(lr_models["China"].coef_, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
coef_s.plot(kind="barh", ax=ax, color=["steelblue" if v >= 0 else "tomato" for v in coef_s])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Linear Regression Coefficients — China")
ax.set_xlabel("Coefficient value")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

print("\nCoefficients (China):")
display(coef_s.round(4).to_frame("coefficient"))

**Most influential feature:** `co2_lag1` typically has the largest absolute coefficient, confirming that last year's emissions is the single strongest predictor of this year's. `years_since_1990` carries a positive coefficient for high-growth countries (China, India), capturing the secular upward trend embedded in the training data. `co2_yoy_pct_change` is often small or slightly negative for stable/declining economies, reflecting mean-reversion in recent rates of change.

Note that Linear Regression coefficients are scale-sensitive — a large coefficient on a low-variance feature (like `co2_yoy_pct_change`) does not necessarily imply high predictive weight. The 2020 COVID dip is an out-of-distribution shock not present in training, so LR over-predicts for that year regardless of which feature dominates.

<a id='35-rf-pc'></a>
### 3.5 Random Forest Regressor — Per Country

Train a separate `RandomForestRegressor(n_estimators=100, random_state=42)` for each
country using only that country’s ~25 training rows and `FEATURES` (no `country_encoded`).
Evaluate per country on the 2019–2023 test set. Results feed into the §3.7 comparison
table to illustrate the effect of training a 100-tree ensemble on insufficient data.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_pc_results = {}
rf_pc_preds   = {}

for country in COUNTRIES:
    train_c = train[train['country'] == country]
    test_c  = test[test['country'] == country]
    if train_c.empty or test_c.empty:
        continue
    rf_pc = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_pc.fit(train_c[FEATURES], train_c[TARGET])
    y_pred = rf_pc.predict(test_c[FEATURES])
    rf_pc_results[country] = {
        'RF-PC MAE':  round(mean_absolute_error(test_c[TARGET], y_pred), 1),
        'RF-PC RMSE': round(np.sqrt(mean_squared_error(test_c[TARGET], y_pred)), 1),
    }
    rf_pc_preds[country] = y_pred

print('Random Forest (per-country) — test set 2019–2023:')
display(pd.DataFrame(rf_pc_results).T)

#### Why we pool all 10 countries for Random Forest

With only ~25 rows per country in the training set (years 1990–2018), training a separate Random Forest per country creates three problems:

1. **Overfitting:** Decision trees built on 25 samples have very high variance — the model memorises training noise rather than learning generalisable patterns.
2. **Unstable bootstrap samples:** Random Forest relies on bootstrapping; with 25 rows the bootstrap samples overlap heavily, making ensemble averaging less effective.
3. **Unreliable feature importance:** Importance scores from a 25-row tree have wide confidence intervals and can be dominated by noise.

By pooling all 10 countries (~250 rows) and adding a `country_encoded` feature, the model can learn cross-country emissions patterns while still distinguishing between countries of different emissions scales.

**Trade-off:** The pooled model learns shared cross-country patterns but cannot capture purely country-specific dynamics (e.g. Germany's structural shift away from coal). This is why Linear Regression remains per-country — 25 rows is adequate for a 6-feature linear model but not for a 100-tree ensemble.

**Key teaching point:** Model complexity must match data availability. A simple model trained on adequate data typically outperforms a complex model trained on insufficient data.

<a id='36-rf-pooled'></a>
### 3.6 Random Forest Regressor — Pooled

Train a single `RandomForestRegressor(n_estimators=100, random_state=42)` on the **pooled dataset of all 10 countries** (~250 training rows) with a `country_encoded` feature (via `LabelEncoder`). Evaluate per country on the 2019–2023 test set. Plot feature importances from the single pooled model as a horizontal bar chart.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder

# Fit encoder on the full COUNTRIES list so all 10 are always known,
# regardless of which countries survive dropna in train
le = LabelEncoder().fit(COUNTRIES)
train = train.copy()
test  = test.copy()
train.loc[:, 'country_encoded'] = le.transform(train['country'])
test.loc[:, 'country_encoded']  = le.transform(test['country'])

# country_encoded is RF-only — do not add to shared FEATURES (used by LR)
RF_FEATURES = FEATURES + ['country_encoded']

# --- Extended training set for RF Pooled (1975+) ---
# Experiment shows RF Pooled MAE improves for all 10 countries when trained
# on 1979–2018 (~40 rows/country) vs the default 1994–2018 (~24 rows/country).
# LR and ETS are unaffected — they use train / df_filtered as before.
_df_ext = (
    pd.read_csv('../data/owid-co2-data.csv',
                usecols=['country', 'year', 'co2'])
      .query("country in @COUNTRIES and year >= 1975")
      .copy()
)
_df_ext.loc[:, 'years_since_1990']     = _df_ext['year'] - 1990
_df_ext.loc[:, 'co2_5yr_rolling_mean'] = (
    _df_ext.groupby('country')['co2'].transform(lambda x: x.rolling(5).mean())
)
_df_ext.loc[:, 'co2_lag1'] = _df_ext.groupby('country')['co2'].shift(1)
_df_ext.loc[:, 'co2_lag2'] = _df_ext.groupby('country')['co2'].shift(2)
_df_ext.loc[:, 'co2_lag3'] = _df_ext.groupby('country')['co2'].shift(3)
_df_ext.loc[:, 'co2_yoy_pct_change'] = (
    _df_ext.groupby('country')['co2'].pct_change() * 100
)
_df_ext.loc[:, 'country_encoded'] = le.transform(_df_ext['country'])
_train_ext = (
    _df_ext[_df_ext['year'] <= TRAIN_CUTOFF]
    .dropna(subset=RF_FEATURES + [TARGET])
)
print(f"RF Pooled extended training: {len(_train_ext)} rows "
      f"({_train_ext['year'].min()}–{_train_ext['year'].max()}) "
      f"across {_train_ext['country'].nunique()} countries")

# Single pooled RF trained on extended dataset (~400 rows, 1979–2018)
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(_train_ext[RF_FEATURES], _train_ext[TARGET])

# Evaluate per country on the standard test set (2019–2023)
rf_results = {}
rf_preds   = {}

for country in COUNTRIES:
    test_c = test[test['country'] == country]
    if test_c.empty:
        continue
    y_pred = rf_model.predict(test_c[RF_FEATURES])
    rf_results[country] = {
        'RF MAE':  round(mean_absolute_error(test_c[TARGET], y_pred), 1),
        'RF RMSE': round(np.sqrt(mean_squared_error(test_c[TARGET], y_pred)), 1),
    }
    rf_preds[country] = y_pred

print('Random Forest (pooled) — test set 2019–2023:')
display(pd.DataFrame(rf_results).T)

# Feature importance — single pooled model
imp_s = pd.Series(rf_model.feature_importances_, index=RF_FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
imp_s.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Random Forest Feature Importances (Pooled Model — All 10 Countries)')
ax.set_xlabel('Importance (mean decrease in impurity)')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()


<a id='37-comparison'></a>
### 3.7 Model Comparison Table

Assemble results for all 10 countries:

```
Country | Baseline MAE | LR MAE | RF-PC MAE | RF MAE | Baseline RMSE | LR RMSE | RF-PC RMSE | RF RMSE | Best Model
```

> **Note:** You will extend this table in §4.6 to add ETS(A,Ad,N), completing a 5-model
> consolidated comparison. RF-PC = Random Forest trained per country (~25 rows);
> RF = Random Forest trained on the pooled dataset (~250 rows).

In [ ]:
df_baseline = pd.DataFrame(baseline_results).T
df_lr       = pd.DataFrame(lr_results).T
df_rf_pc    = pd.DataFrame(rf_pc_results).T
df_rf       = pd.DataFrame(rf_results).T

df_comparison = df_baseline.join(df_lr).join(df_rf_pc).join(df_rf)

mae_cols = ["Baseline MAE", "LR MAE", "RF-PC MAE", "RF MAE"]
df_comparison["Best Model"] = (
    df_comparison[mae_cols].idxmin(axis=1)
    .str.replace(" MAE", "", regex=False)
)

display(df_comparison.sort_index())

**Conclusion:** Linear Regression outperforms all other models for all 10 countries — the Best Model column is LR across the board. LR reduces MAE substantially over the baseline in every case (Russia −94%, Australia −88%, Brazil −81%), demonstrating that each country's emissions trend is well-captured by a simple linear combination of its own lag and rolling-mean features.

The RF Per-Country results (§3.5) are the most striking finding: per-country RF underperforms the naive baseline for **9 of 10 countries** — China (RF-PC MAE 1328.2 vs Baseline 323.7), USA (365.1 vs 198.0), India (318.8 vs 162.5), Japan (147.9 vs 36.5), Germany (127.8 vs 41.3), UK (60.2 vs 18.2), Russia (39.5 vs 37.8), South Africa (16.8 vs 13.0), Australia (13.7 vs 5.6). Brazil is the sole exception (RF-PC MAE 15.1 vs Baseline 16.5 — marginal improvement). A 100-tree ensemble trained on 25 samples memorises training noise completely and fails to generalise even to the next year's value.

The pooled RF (§3.6, ~250 rows) improves on per-country RF for 9 of 10 countries — but still underperforms the naive baseline for five of them (China, USA, Japan, Germany, UK). More training data helps but cannot overcome the loss of country-specific trend structure when countries with divergent trajectories (rapid growth vs sustained decline) are mixed together. Both RF variants also cannot extrapolate beyond the training range, amplifying errors during the 2020 COVID-19 out-of-distribution dip.

The RF-PC vs RF comparison directly validates the mandatory limitations cell (§3.6): model complexity must match data availability, and pooling is the right design choice for ensemble methods on small per-country datasets. The consolidated five-model comparison (adding ETS MAE/RMSE) will be completed in §4.6 after Week 4.

---
> ```
> git add notebook/ghg_analysis.ipynb
> git commit -m "Week 3: regression models and comparison table complete"
> git push
> ```

<a id='38-regression-forecast'></a>
### 3.8 Regression Model Forecasts to 2043

Extend the trained **RF Pooled** model beyond the 2019–2023 test set using **recursive (iterative) forecasting**: predict one year at a time, feed each prediction back as the next step's lag feature, and repeat through 2043.

> **Why RF, not LR?** Linear Regression can predict negative CO₂ values for countries with declining trends (UK, Germany, Japan). When negative values feed back as `co2_lag1/2/3`, subsequent predictions diverge rapidly. Random Forest cannot extrapolate beyond its training range, so its recursive predictions stay bounded — making it far more stable at long horizons despite LR's superior 5-year holdout performance.


In [ ]:
def build_forecast_features(history, yr):
    """Build one FEATURES row for forecast year yr using history dict {year: co2}."""
    recent = [history[yr - k] for k in range(1, 6) if (yr - k) in history]
    rolling_mean = np.mean(recent) if recent else np.nan
    prev, prev2 = history.get(yr - 1), history.get(yr - 2)
    yoy = ((prev - prev2) / prev2 * 100) if (prev is not None and prev2) else 0.0
    return {
        'years_since_1990':     yr - 1990,
        'co2_5yr_rolling_mean': rolling_mean,
        'co2_lag1':             history.get(yr - 1, np.nan),
        'co2_lag2':             history.get(yr - 2, np.nan),
        'co2_lag3':             history.get(yr - 3, np.nan),
        'co2_yoy_pct_change':   yoy,
    }

fc_years = np.arange(TRAIN_CUTOFF + 1, FORECAST_END + 1)  # 2019–2043

rf_forecasts_fc = {}

for country in COUNTRIES:
    base_hist = dict(zip(
        df_filtered[(df_filtered['country'] == country) & (df_filtered['year'] <= TRAIN_CUTOFF)]['year'],
        df_filtered[(df_filtered['country'] == country) & (df_filtered['year'] <= TRAIN_CUTOFF)]['co2'],
    ))

    rf_hist = dict(base_hist)
    rf_preds_fc = {}
    c_enc = le.transform([country])[0]

    for yr in fc_years:
        row_rf = {**build_forecast_features(rf_hist, yr), 'country_encoded': c_enc}
        rf_pred = rf_model.predict(pd.DataFrame([row_rf], columns=RF_FEATURES))[0]
        rf_preds_fc[yr] = rf_pred
        rf_hist[yr] = rf_pred

    rf_forecasts_fc[country] = pd.Series(rf_preds_fc)

fig, axes = plt.subplots(5, 2, figsize=(10, 20))
axes_flat = axes.flatten()

for ax, country in zip(axes_flat, COUNTRIES):
    series  = df_filtered[df_filtered['country'] == country].sort_values('year').set_index('year')['co2']
    test_c  = test[test['country'] == country].set_index('year')['co2']
    train_s = series[series.index <= TRAIN_CUTOFF]

    ax.plot(train_s.index,                  train_s,                  color='steelblue',  lw=1.5,          label='Train actuals')
    ax.plot(test_c.index,                   test_c,                   color='darkorange', lw=1.5,          label='Holdout actuals')
    ax.plot(rf_forecasts_fc[country].index, rf_forecasts_fc[country], color='purple',     lw=1.5,          label='RF Pooled Forecast')
    ax.set_title(country, fontsize=9)
    ax.set_xlabel('Year', fontsize=7)
    ax.set_ylabel('CO\u2082 (MtCO\u2082)', fontsize=7)
    ax.tick_params(labelsize=7)
    ax.grid(axis='y', alpha=0.3)

axes_flat[0].legend(fontsize=6, loc='upper left')
plt.suptitle('RF Pooled Recursive Forecasts to 2043 — All 10 Countries', fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


---
<a id='week4'></a>
# Week 4 — Time-Series Forecasting with ETS(A,Ad,N) — Holt's Damped Trend

**Learning Objective:** Apply ETS(A,Ad,N) to generate multi-year emissions forecasts with confidence intervals; understand why a damped trend model is well-suited to long-range annual emissions data.

> **Model rationale (v3):** ARIMA has been replaced by ETS(A,Ad,N) — Holt's Damped Trend. The damping parameter φ prevents unbounded trend extrapolation over a 20-year horizon, which is more physically realistic for emissions data — particularly for countries with documented slowdowns (UK, Germany). It also requires fewer modelling decisions: no stationarity testing, no ACF/PACF order selection.

> **Index note:** Work with integer-indexed pandas Series (indexed by `year`). `fit.forecast(steps)` returns a `pd.Series` whose index continues from the last training year (2018), so the first forecast value has index 2019.

<a id='41-concept'></a>
### 4.1 Concept Introduction

ETS(A,Ad,N) is a **state space model** that updates three components recursively from each new observation — no feature matrix is required. It differs from the Week 3 regression models in that it uses only the target series itself (CO₂ over time), not engineered lag or rolling-mean features.

| Component | Symbol | What it means |
|-----------|--------|---------------|
| Error | E = A (Additive) | Forecast errors are added to the state, not multiplied |
| Trend | T = Ad (Additive Damped) | Trend grows additively but decays toward zero via damping parameter φ (0 < φ < 1) |
| Seasonality | S = N (None) | Annual data has no within-year seasonal cycle to model |

**Why ETS(A,Ad,N) is appropriate for annual emissions data:**
- No within-year seasonality to model — unlike monthly energy data, annual CO₂ has no calendar cycle
- Damped trend prevents unbounded long-range projections — unlike a unit-root ARIMA, which extrapolates a constant trend indefinitely; after 20 years an undamped model produces physically implausible values
- Works reliably with ~30 data points and has fewer free parameters than ARIMA (no order selection, no stationarity testing required)
- Physically sensible: emissions in real economies tend to slow, plateau, or gradually reverse as energy efficiency improves and policy constraints tighten — not grow at a constant rate indefinitely

**What each fitted parameter represents:**

- **α (smoothing level):** weight given to the most recent observation vs the smoothed history. High α → the model is highly responsive to recent data and "forgets" older observations quickly. Low α → the level estimate is a long-run average.
- **β\* (smoothing trend):** weight given to the most recent trend estimate. High β\* → the trend reacts quickly to recent slope changes. Near zero → the trend is stable and slow to update.
- **φ (damping):** controls how quickly the trend decays toward zero over the forecast horizon. φ near 1 → near-linear extrapolation (trend persists). φ near 0 → trend fades out rapidly and the forecast flattens to a constant level. For most countries φ will be estimated close to 1 by the optimiser, reflecting that emissions trends have been persistent over 1990–2018, but the damping still prevents indefinite linear extrapolation over a 20-year horizon.

<a id='42-model-fitting'></a>
### 4.2 Model Fitting

For each of the 10 countries, fit `ExponentialSmoothing` on the **1990–2018 training series**:

```python
from statsmodels.tsa.holtwinters import ExponentialSmoothing

model = ExponentialSmoothing(
    train_co2,        # pandas Series indexed by integer year
    trend='add',
    damped_trend=True,
    seasonal=None
)
fit = model.fit(optimized=True)
```

After fitting, for **at least 3 countries**:
- Print **α** (`fit.params['smoothing_level']`), **β\*** (`fit.params['smoothing_trend']`), and **φ** (`fit.params['damping_trend']`)
- Store each `fit` object in a dictionary keyed by country name for use in §4.3

**✏️ Interpret:** For the 3 countries you printed, what does a high vs low φ value imply about that country's projected emissions trajectory?

*Write your interpretation here.*

In [ ]:
import warnings
from statsmodels.tools.sm_exceptions import ValueWarning
from statsmodels.tsa.holtwinters import ExponentialSmoothing

PLOT_COUNTRIES_ETS = ['China', 'United Kingdom', 'India']

ets_fits = {}

with warnings.catch_warnings():
    warnings.simplefilter('ignore', ValueWarning)
    for country in COUNTRIES:
        series = (
            df_filtered[
                (df_filtered['country'] == country) &
                (df_filtered['year'] <= TRAIN_CUTOFF)
            ]
            .sort_values('year')
            .set_index('year')['co2']
        )
        model = ExponentialSmoothing(series, trend='add', damped_trend=True, seasonal=None)
        ets_fits[country] = model.fit(optimized=True)

print(f"{'Country':<22}  {'α (level)':>10}  {'β* (trend)':>10}  {'φ (damping)':>11}")
print('-' * 58)
for country in PLOT_COUNTRIES_ETS:
    p = ets_fits[country].params
    print(
        f"{country:<22}  "
        f"{p['smoothing_level']:>10.4f}  "
        f"{p['smoothing_trend']:>10.4f}  "
        f"{p['damping_trend']:>11.4f}"
    )


<a id='43-forecasting'></a>
### 4.3 Forecasting to 2043

For each of the 10 countries:
1. Generate out-of-sample forecasts from **2019 to `FORECAST_END`** (2043).
2. Include **95% confidence intervals** via simulation.
3. Produce a forecast plot per country showing four elements with distinct colours:
   - Historical actuals 1990–2018
   - In-sample fitted values
   - 2019–2023 holdout actuals overlaid (for visual validation)
   - Out-of-sample forecast 2019–2043 with CI shading
4. Store each country's forecast Series and CI bounds in a dict for use in §4.5 and §5.2.

**Reference:**
```python
steps = FORECAST_END - TRAIN_CUTOFF   # = 25 (covers 2019–2043)
fc_mean = fit.forecast(steps)         # pd.Series indexed 2019…2043

# Confidence intervals via simulation (HoltWintersResults has no get_forecast)
sim = fit.simulate(nsimulations=steps, repetitions=1000, error='add')
ci_lower = sim.quantile(0.025, axis=1)
ci_upper = sim.quantile(0.975, axis=1)

# Store for §4.5 and §5.2
forecasts[country] = {'mean': fc_mean, 'ci_lower': ci_lower, 'ci_upper': ci_upper}
```

> Use `ax.fill_between(fc_mean.index, ci_lower, ci_upper, alpha=0.2)` for CI shading.

In [ ]:
import warnings
from statsmodels.tools.sm_exceptions import ValueWarning

forecasts = {}
steps = FORECAST_END - TRAIN_CUTOFF   # 25 steps: 2019–2043

fig, axes = plt.subplots(5, 2, figsize=(10, 20))
axes_flat = axes.flatten()

for ax, country in zip(axes_flat, COUNTRIES):
    fit    = ets_fits[country]
    series = (
        df_filtered[df_filtered['country'] == country]
        .sort_values('year').set_index('year')['co2']
    )
    test_c = test[test['country'] == country].set_index('year')['co2']

    with warnings.catch_warnings():
        warnings.simplefilter('ignore', ValueWarning)
        warnings.simplefilter('ignore', FutureWarning)
        fc_mean = fit.forecast(steps)
        sim     = fit.simulate(nsimulations=steps, repetitions=1000, error='add')
    ci_lo   = sim.quantile(0.025, axis=1)
    ci_hi   = sim.quantile(0.975, axis=1)

    forecast_years = np.arange(TRAIN_CUTOFF + 1, FORECAST_END + 1)
    fc_mean.index = forecast_years
    ci_lo.index   = forecast_years
    ci_hi.index   = forecast_years

    forecasts[country] = {'mean': fc_mean, 'ci_lower': ci_lo, 'ci_upper': ci_hi}

    train_s = series[series.index <= TRAIN_CUTOFF]
    fitted  = fit.fittedvalues

    ax.plot(train_s.index, train_s,  color='steelblue',  lw=1.5, label='Train actuals')
    ax.plot(fitted.index,  fitted,   color='steelblue',  lw=1,   ls='--', alpha=0.6, label='Fitted')
    ax.plot(test_c.index,  test_c,   color='darkorange', lw=1.5, label='Holdout actuals')
    ax.plot(fc_mean.index, fc_mean,  color='green',      lw=1.5, label='Forecast')
    ax.fill_between(fc_mean.index, ci_lo, ci_hi, color='green', alpha=0.15, label='95% CI')
    ax.set_title(country, fontsize=9)
    ax.set_xlabel('Year', fontsize=7)
    ax.set_ylabel('CO₂ (MtCO₂)', fontsize=7)
    ax.tick_params(labelsize=7)
    ax.grid(axis='y', alpha=0.3)

axes_flat[0].legend(fontsize=6, loc='upper left')
plt.suptitle('ETS(A,Ad,N) Forecasts to 2043 — All 10 Countries', fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


<a id='44-trend'></a>
### 4.4 Trend Interpretation

The three showcase countries — China, United Kingdom, and India — illustrate the three qualitatively different trajectory types visible across the 10-country panel.

---

**China** (α = 0.9955, β\* = 0.7040, φ = 0.9710)

China's near-unity α means the model almost entirely discards history when updating the level: each new observation dominates. The high β\* (0.704) lets the trend react strongly to year-over-year acceleration or deceleration. The damping factor φ = 0.971 is moderate — compounded over the 25-year horizon, annual trend increments shrink to roughly (0.971²⁵ ≈ 0.48) of their 2018 value, so growth continues but at progressively slower pace. The model projects emissions rising from 10,897 MtCO₂ (2020) to 14,932 MtCO₂ by 2040 (+37 %). This is broadly consistent with China's stated "peak before 2030" target: the damped trend structurally decelerates through the 2020s without requiring an explicit policy break-point. The 95 % CI is wide in absolute terms by 2040 (China's scale amplifies forecast uncertainty), and the ETS MAE on the 2019–2023 holdout is 234.5 MtCO₂ — better than both RF variants for China but behind Linear Regression (64.3 MtCO₂).

---

**United Kingdom** (α = 0.5215, β\* = 0.3995, φ = 0.9757)

The UK's moderate α and lower β\* produce a smoother, less reactive fit: recent structural shifts in the energy mix (coal phase-out, gas decline, renewables growth) are incorporated gradually rather than spike-by-spike. φ = 0.976 is similar to China's, so the declining trend also attenuates — by 2040, annual reductions are roughly (0.976²⁵ ≈ 0.54) of their 2018 rate — but because the underlying trend is *negative*, damping means the decline slows rather than stopping. The projection falls from 326.3 MtCO₂ (2020) to 109.8 MtCO₂ by 2040 (−66 %), broadly in line with the UK's legally binding 78 % reduction target by 2035 under the Climate Change Act. Absolute CI widths are the narrowest of the three countries, reflecting the lower emission scale and more predictable policy trajectory. ETS achieves the lowest MAE here (12.0 MtCO₂) — the only country where ETS outperforms the Baseline and both RF variants, though LR still leads (8.9 MtCO₂).

---

**India** (α = 0.6392, β\* = 0.6388, φ = 0.9950)

India's φ = 0.995 is nearly 1 — effectively no damping over any practical horizon. After 25 steps the trend has decayed to only (0.995²⁵ ≈ 0.88) of its starting value, meaning India's growth rate in the model is almost constant through 2043. This produces the most aggressive projection: emissions nearly double from 2,422.7 MtCO₂ (2020) to 5,071.0 MtCO₂ by 2040 (+109 %), making India the second-largest emitter in the forecast by 2030. The near-undamped trend also generates the widest 95 % CI of the three countries by 2040: if India's rapid renewable-energy buildout bends the emissions curve before the model horizon, the actual outcome could fall well below the central forecast. The ETS holdout MAE of 189.0 MtCO₂ reflects the difficulty of forecasting high-growth economies with a purely statistical model that cannot anticipate policy discontinuities.


<a id='45-forecast-table'></a>
### 4.5 Forecast Summary Table

Extract point forecast values for **2030, 2035, and 2040** from each country's `forecasts[country]['mean']` (the `pd.Series` stored in §4.3). Look up the **2020 actual** CO₂ from `df_filtered`. Build a summary DataFrame:

```
Country | 2030 Forecast (MtCO₂) | 2035 Forecast | 2040 Forecast | 2020 Actual | % Change 2020→2040
```

Round forecast values to 1 decimal place and sort by 2040 forecast (descending).

In [ ]:
rows = []
for country in COUNTRIES:
    fc_mean     = forecasts[country]['mean']
    actual_2020 = df_filtered[
        (df_filtered['country'] == country) & (df_filtered['year'] == 2020)
    ]['co2'].values[0]
    fc_2040     = round(fc_mean.loc[2040], 1)
    pct_change  = round((fc_2040 - actual_2020) / actual_2020 * 100, 1)
    rows.append({
        'Country':                country,
        '2030 Forecast (MtCO₂)': round(fc_mean.loc[2030], 1),
        '2035 Forecast':          round(fc_mean.loc[2035], 1),
        '2040 Forecast':          round(fc_mean.loc[2040], 1),
        '2020 Actual':            round(actual_2020, 1),
        '% Change 2020→2040':     pct_change,
    })

df_forecast_summary = (
    pd.DataFrame(rows)
    .set_index('Country')
    .sort_values('2040 Forecast', ascending=False)
)
display(df_forecast_summary)


<a id='46-validation'></a>
### 4.6 Model Validation

1. Compute MAE and RMSE of ETS(A,Ad,N) forecasts against the **2019–2023 holdout actuals** for each country.
2. **Extend the §3.7 comparison table** to add `ETS MAE` and `ETS RMSE`, completing the five-model consolidated comparison:

```
Country | Baseline MAE | LR MAE | RF-PC MAE | RF MAE | ETS MAE | Baseline RMSE | LR RMSE | RF-PC RMSE | RF RMSE | ETS RMSE | Best Model
```


In [ ]:
ets_results = {}

for country in COUNTRIES:
    test_c  = test[test['country'] == country]
    if test_c.empty:
        continue
    fc_mean = forecasts[country]['mean']
    y_pred  = fc_mean.loc[test_c['year'].values].values
    y_true  = test_c[TARGET].values
    ets_results[country] = {
        'ETS MAE':  round(mean_absolute_error(y_true, y_pred), 1),
        'ETS RMSE': round(np.sqrt(mean_squared_error(y_true, y_pred)), 1),
    }

df_ets = pd.DataFrame(ets_results).T
df_comparison_full = df_comparison.join(df_ets)

mae_cols_full = ["Baseline MAE", "LR MAE", "RF-PC MAE", "RF MAE", "ETS MAE"]
df_comparison_full["Best Model"] = (
    df_comparison_full[mae_cols_full].idxmin(axis=1)
    .str.replace(" MAE", "", regex=False)
)

print("Five-model comparison — Naive Baseline · LR · RF Per-Country · RF Pooled · ETS(A,Ad,N):")
display(df_comparison_full.sort_index())


---
## Week 6 — Data Exports for Streamlit Dashboard

Run the two cells below after completing §4.6 to generate the CSV files
required by `app.py`:
- `data/ets_forecasts.csv` — ETS forecast means and 95% CI bounds (2019–2043)
- `data/model_comparison.csv` — Five-model MAE/RMSE comparison table


In [ ]:
# Export ETS forecasts to CSV for Streamlit dashboard
rows = []
for country in COUNTRIES:
    for yr, mean_val in forecasts[country]['mean'].items():
        rows.append({
            'country':  country,
            'year':     int(yr),
            'mean':     round(float(mean_val), 3),
            'ci_lower': round(float(forecasts[country]['ci_lower'].loc[yr]), 3),
            'ci_upper': round(float(forecasts[country]['ci_upper'].loc[yr]), 3),
        })
df_ets_export = pd.DataFrame(rows)
df_ets_export.to_csv('../data/ets_forecasts.csv', index=False)
print(f"Exported ets_forecasts.csv — {len(df_ets_export)} rows")
display(df_ets_export.head())

In [ ]:
# Export five-model comparison table to CSV for Streamlit dashboard
df_comparison_full.reset_index().rename(columns={'index': 'country'}).to_csv(
    '../data/model_comparison.csv', index=False
)
print("Exported model_comparison.csv")
display(df_comparison_full)

**Discussion — ETS(A,Ad,N) vs the regression models (2019–2023 holdout):**

Linear Regression remains the best model for all 10 countries on the 2019–2023 holdout, a result that holds even after adding ETS to the comparison. ETS outperforms the naïve Baseline for only 2 of 10 countries — China (ETS MAE 234.5 vs Baseline 323.7) and the United Kingdom (12.0 vs 18.2) — and outperforms the pooled Random Forest for 2 of 10 (China and UK only — Japan ETS MAE 109.5 exceeds RF Pooled 39.8, and USA ETS 231.2 exceeds RF Pooled 225.1). Against the per-country RF, ETS wins 6 of 10 (China, Germany, India, Japan, UK, United States). This middling performance is expected: the five-year holdout (2019–2023) coincides with the COVID-19 dip and partial rebound, which are discrete shocks that a trend-smoothing model cannot anticipate. LR's simple linear relationship with `years_since_1990` happens to interpolate through this turbulence well because it has no "memory" of pre-COVID momentum to defend. ETS's strength — the damped trend that prevents unbounded extrapolation — is more visible at longer horizons (§4.5), where the regression models' linear extrapolation becomes physically implausible (e.g. predicting the UK's linear decline reaches zero CO₂ before 2050). For the purpose of 25-year scenario analysis, ETS(A,Ad,N) is the appropriate choice despite its inferior short-horizon holdout scores.


---
> ```
> git add notebook/ghg_analysis.ipynb
> git commit -m "Week 4: ETS(A,Ad,N) Holt Damped forecasting complete"
> git push
> ```

---
<a id='week5'></a>
# Week 5 — Scenario Analysis *(Optional — Complete Only if Time Permits)*

> ⚠️ **Proceed only if Weeks 3 and 4 are fully complete, documented, and committed to GitHub.** If time is limited, skip to Week 6.

**Learning Objective:** Build a what-if scenario module to simulate emissions policy impact.


<a id='51-design'></a>
### 5.1 Scenario Design

| Scenario | Label | Applied from 2025 |
|----------|-------|-------------------|
| A | Business as Usual (BAU) | ETS(A,Ad,N) forecast unchanged |
| B | Moderate Mitigation | −2% per year applied to BAU |
| C | Aggressive Mitigation | −5% per year applied to BAU |

**✏️ Explain** the basis and limitations of each scenario. These are illustrative — what real-world conditions would each require?

*Write your explanation here.*

<a id='52-calc'></a>
### 5.2 Scenario Calculation

Compute annual projections from 2025 to 2040 for each country and scenario.

```python
# Outline — implement the details
# bau_2025_2040 = ETS(A,Ad,N) forecast values for 2025–2040 (from §4.3 fc_df)
# years_elapsed = np.arange(len(bau_2025_2040))  # 0, 1, 2, ...
# moderate   = bau_2025_2040 * (1 - 0.02) ** years_elapsed
# aggressive = bau_2025_2040 * (1 - 0.05) ** years_elapsed
```

Store in a tidy long-format DataFrame: `country, year, scenario, co2_projected`.
Save as `data/scenario_projections.csv`.

In [ ]:
# Compute BAU, Moderate, Aggressive scenarios for all 10 countries
# Save to data/scenario_projections.csv


<a id='53-viz'></a>
### 5.3 Scenario Visualisations

Per-country: one chart overlaying all 3 scenarios (2020–2040) + grey historical reference (1990–2024) + dashed horizontal line at 1990 emissions level.

**Color convention:** BAU = blue | Moderate = orange | Aggressive = green

Also produce a **global aggregate chart**: sum all 10 countries per scenario per year.


In [ ]:
# Per-country overlay charts
# Global aggregate chart


<a id='54-impact'></a>
### 5.4 Impact Summary

Compute cumulative CO₂ per country per scenario (2025–2040 sum). Plot as a grouped bar chart.


In [ ]:
# Cumulative CO₂ 2025–2040 per country per scenario
# Grouped bar chart


**✏️ Interpretation (3–5 sentences):** Which countries benefit most from aggressive mitigation in absolute and percentage terms?

*Write your interpretation here.*


<a id='55-sensitivity'></a>
### 5.5 Parameter Sensitivity *(Stretch Goal)*

For one country of your choice, show how cumulative 2025–2040 emissions change as the Moderate reduction rate varies from 1% to 4% in 1% steps.

| Reduction Rate | Cumulative CO₂ 2025–2040 (MtCO₂) | % Reduction vs BAU |
|---------------|----------------------------------|-------------------|
| 1% | | |
| 2% | | |
| 3% | | |
| 4% | | |


In [ ]:
# Sensitivity table for one selected country


---
> ```
> git add notebook/ghg_analysis.ipynb data/scenario_projections.csv
> git commit -m "Week 5: scenario analysis complete"
> git push
> ```


---
<a id='week6'></a>
# Week 6 — Notebook Finalisation and Optional Streamlit Dashboard

**Learning Objective:** Finalise the notebook to professional standards; optionally build the dashboard.

### 6.1 Notebook Finalisation Checklist

- [ ] Every section has an **opening markdown** (what + why) and a **closing summary** (key findings)
- [ ] **Table of Contents** at the top links to all completed sections
- [ ] All charts: consistent colour schemes, font sizes, and labelling
- [ ] All debugging `print()` statements removed
- [ ] **Kernel → Restart & Run All** — zero errors top to bottom

### 6.2 Streamlit App *(Stretch Goal)*

Open `app.py` (already scaffolded in this repository) and fill in the `# TODO:` sections using Plotly Express charts and your generated data files.

> **Prerequisite:** `data/ghg_features.csv` must exist. The app gracefully handles missing files.

```bash
streamlit run app.py
```

**Minimum interactivity required:**
- `st.selectbox` for country selection
- `st.multiselect` for gas type
- `st.radio` for scenario selection (if Week 5 complete)
- All charts: **Plotly Express** (not Matplotlib)

### 6.3 GitHub Repository Requirements

- [ ] `notebook/ghg_analysis.ipynb` — all cells run cleanly
- [ ] `app.py` — runs without errors
- [ ] `data/ghg_features.csv` and (if applicable) `data/scenario_projections.csv`
- [ ] `requirements.txt` — complete
- [ ] `README.md` — team names filled in, instructions verified

---
> ```
> git add .
> git commit -m "Week 6: notebook finalised, Streamlit app added"
> git push
> ```


---
<a id='week7'></a>
# Week 7 — Final Presentation

**Learning Objective:** Deliver a structured 1-hour presentation demonstrating the full analytical workflow.

> *The project report format and formal submission requirements are provided separately.*

### Presentation Outline (60 minutes)

| Slot | Duration | Content |
|------|----------|---------|
| Introduction | 5 min | Project objective, team intro, dataset overview |
| EDA Findings | 10 min | 3 charts + written observations |
| Feature Engineering | 5 min | Key features, rolling mean and lag rationale |
| Model Comparison | 15 min | Live notebook — Baseline → LR → RF → ETS(A,Ad,N), consolidated table |
| Forecasts | 10 min | 3 country forecast plots, summary table, trend interpretation |
| Scenario Analysis | 5 min | Key finding *(if completed)* |
| Streamlit Demo | 5 min | Live app run *(if completed)* |
| Key Takeaways & Q&A | 5 min | Limitations, learnings, next steps |

### Final Submission Checklist

- [ ] Jupyter Notebook: fully documented, all cells run top to bottom without errors
- [ ] Streamlit app *(if built)*: runs without errors, `requirements.txt` complete
- [ ] GitHub repository link **shared before the session**
- [ ] Presentation slides **shared before the session**
- [ ] Project report: submitted in the required report format